In [ ]:
---
layout: post 
title: Gamify Exploration 
description: A game design lesson on sprite sheets, chase logic, sprite swapping, and NPC interaction systems
permalink: /characters-lesson/
hide: true
toc: false
author: Lance Oberiano, Yiming Yin, Arjun Ganesh
---


In [ ]:
%%js

// GAME_RUNNER: CSSE Zone Game | hide_edit: true,  width: 100%, height: 500px

import GameControl from './assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelAquaticGameLevel from './assets/js/projects/characters/levels/GameLevelAquaticGameLevel.js';
import GameLevelSeek from './assets/js/projects/characters/levels/GameLevelSeek.js';
import GameLevelBasketball from './assets/js/projects/characters/levels/GameLevelBasketball.js';


export const gameLevelClasses = [GameLevelAquaticGameLevel, GameLevelSeek, GameLevelBasketball,];
export { GameControl };

# Basketball — How the Chase Logic Works
A line-by-line breakdown of how Kirby hunts the player in `GameLevelBasketball.js`.

---

## What Is Chase Logic?

Chase logic is code that runs every frame and moves an enemy closer to the player. It does not plan ahead, it does not remember the past — it just looks at where the player is right now and takes one step toward them.

In the Basketball level, Kirby chases your astro player across the court. Every frame, the `update()` method is called, and this is where the chase happens.

---

## Step 1 — Find Both Objects

Before Kirby can chase anyone, the code has to find both game objects in the scene.

```js
const player = this.findById('BasketballPlayer');
const lebron = this.findById('LeBron');
if (!player || !lebron) return;
```

`findById` searches `gameEnv.gameObjects` — the live list of everything in the level — and returns the one whose `spriteData.id` matches. If either is missing (still loading, already destroyed), the `return` bails out early so nothing crashes.

The rule: always guard with an early return before touching positions. A missing object has no `.position` to read.

---

## Step 2 — Calculate the Direction Vector

This is the heart of chase logic. A direction vector is just the difference between where the player is and where Kirby is.

```js
const dx = player.position.x - lebron.position.x;
const dy = player.position.y - lebron.position.y;
const dist = Math.hypot(dx, dy);
if (dist < 1) return;
```

`dx` and `dy` are the raw horizontal and vertical gaps. If the player is to the right and below Kirby, `dx` is positive and `dy` is positive.

`Math.hypot(dx, dy)` is the straight-line distance between them — the same as `Math.sqrt(dx*dx + dy*dy)`, just written more cleanly.

The `if (dist < 1) return` guard prevents a divide-by-zero on the next step. If Kirby is essentially on top of the player, there is nothing to calculate.

---

## Step 3 — Normalize and Move

Raw `dx` and `dy` are different every frame depending on how far apart the two are. You need to strip out the distance so you get a clean direction, then scale it to the speed you want.

```js
const speed = Math.min(2.1 + this.currentTime * 0.03, 2.8);
lebron.position.x += (dx / dist) * speed;
lebron.position.y += (dy / dist) * speed;
```

`dx / dist` and `dy / dist` give you the normalized direction — a vector whose length is exactly 1, pointing from Kirby toward the player. Multiply by `speed` and you get exactly that many pixels of movement per frame, no matter how far apart they are.

Why the speed formula? Kirby starts at `2.1` pixels per frame and grows `0.03` faster for every second the player survives. `Math.min(..., 2.8)` caps it so the game stays winnable. The longer you last, the harder it gets — but it never becomes impossible.

```
speed at  0s = 2.1
speed at 10s = 2.4
speed at 23s = 2.79  (nearly capped)
speed at 24s = 2.8   (capped — stays here)
```

---

## Step 4 — Clamp to the Court

After moving, the code makes sure Kirby cannot walk off the edge of the canvas.

```js
lebron.position.x = Math.max(0, Math.min(lebron.position.x, this.gameEnv.innerWidth  - (lebron.width  || 0)));
lebron.position.y = Math.max(0, Math.min(lebron.position.y, this.gameEnv.innerHeight - (lebron.height || 0)));
```

`Math.max(0, ...)` prevents Kirby from going past the left or top edge. `Math.min(..., innerWidth - width)` prevents him from going past the right or bottom edge. Without this, Kirby would eventually slide off-screen chasing the player into a corner.

---

## Step 5 — Update Facing Direction

The sprite needs to know which way it is walking so it draws the right animation frame.

```js
if (Math.abs(dx) > Math.abs(dy)) {
  lebron.direction = dx >= 0 ? 'right' : 'left';
} else {
  lebron.direction = dy >= 0 ? 'down' : 'up';
}
```

This compares which axis has the bigger gap. If Kirby is moving more horizontally than vertically, face left or right. If more vertical, face up or down. It is a simple dominant-axis check — no diagonal sprites needed.

---

## Step 6 — Check for Collision

Once Kirby has moved, the code checks whether he has caught the player.

```js
if (this.isHitboxCollision(player, lebron)) {
  this.caught = true;
  this.caughtAt = now;
  this.bestTime = Math.max(this.bestTime, this.currentTime);
  // ...
  this.showCaughtMessage();
}
```

`isHitboxCollision` does an AABB check — it shrinks each sprite to its hitbox rectangle and tests whether those rectangles overlap:

```js
getHitboxRect(obj) {
  const widthReduction  = obj.width  * 0.2;
  const heightReduction = obj.height * 0.2;
  return {
    left:   obj.position.x + widthReduction,
    right:  obj.position.x + obj.width  - widthReduction,
    top:    obj.position.y + heightReduction,
    bottom: obj.position.y + obj.height
  };
}
```

The 20% reduction on each side means the hitbox is smaller than the sprite image. This gives the player a little grace — a near-miss that looks like a catch visually does not register as a catch in code.

---

## Stun: Interrupting the Chase

Kirby's chase can be paused by shooting a basketball at him (press `E`). The stun is just a timestamp check at the top of `update()`:

```js
if (now < this.lebronStunUntil) {
  lebron.velocity.x = 0;
  lebron.velocity.y = 0;
  return;
}
```

`lebronStunUntil` is set to `now + 3000` (3 seconds) when a projectile hits. While the current time is before that timestamp, the function returns before any chase code runs. Kirby freezes completely for 3 seconds, then resumes chasing as normal.

The stun does not reset the direction vector or break anything — the next frame after the stun expires, the full chase loop runs as if nothing happened.

---

## The Full Chase Loop, All Together

Here is the complete `update()` chase section with comments showing what each part does:

```js
update() {
  const player = this.findById('BasketballPlayer');
  const lebron = this.findById('LeBron');
  if (!player || !lebron) return;          // Step 1 — guard

  const now = performance.now();

  if (now < this.lebronStunUntil) {        // Stun check — skip chase if stunned
    lebron.velocity.x = 0;
    lebron.velocity.y = 0;
    return;
  }

  const dx   = player.position.x - lebron.position.x;  // Step 2 — direction
  const dy   = player.position.y - lebron.position.y;
  const dist = Math.hypot(dx, dy);
  if (dist < 1) return;

  const speed = Math.min(2.1 + this.currentTime * 0.03, 2.8);  // Step 3 — speed curve
  lebron.position.x += (dx / dist) * speed;                     // Step 3 — normalize & move
  lebron.position.y += (dy / dist) * speed;

  lebron.position.x = Math.max(0, Math.min(lebron.position.x,   // Step 4 — clamp
    this.gameEnv.innerWidth  - (lebron.width  || 0)));
  lebron.position.y = Math.max(0, Math.min(lebron.position.y,
    this.gameEnv.innerHeight - (lebron.height || 0)));

  if (Math.abs(dx) > Math.abs(dy)) {                             // Step 5 — facing
    lebron.direction = dx >= 0 ? 'right' : 'left';
  } else {
    lebron.direction = dy >= 0 ? 'down' : 'up';
  }

  if (this.isHitboxCollision(player, lebron)) {                  // Step 6 — collision
    this.caught = true;
    // ...
  }
}
```

The rule: every frame, recalculate. There is no stored path, no remembered target. The enemy just looks at where the player is right now and takes one step. That one-step-per-frame pattern, repeated 60 times a second, is what creates smooth, responsive chase behavior.
